# Joan Tryhard

### Imports

In [2]:
import pandas as pd
import sklearn
import imblearn
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

### Get Data and Preprocess

In [3]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd


train = pd.read_csv("data/train_dataset_processed.csv")
test = pd.read_csv("data/test_dataset_processed.csv")

languages = train['language'].unique()

enc = OneHotEncoder(sparse_output=True)
language_encoded = enc.fit_transform(train[['language']])
language_df = pd.DataFrame(language_encoded.toarray(),
                        columns=enc.get_feature_names_out(['language']),
                        index=train.index)
train = pd.concat([train.drop(columns=['language']), language_df], axis=1)

language_encoded_test = enc.transform(test[['language']])
language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                columns=enc.get_feature_names_out(['language']),
                                index=test.index)
test = pd.concat([test.drop(columns=['language']), language_df_test], axis=1)

#Prepare data and labels
X_train = train.drop(columns=['root'])
y_train = train['root']
X_test = test

In [4]:
X_train
y_train
X_test

,sentence_id,node,length,degree,avg_neighbor_deg,degree_squared,degree_diff,clustering,local_degree_ratio,max_neighbor_degree,...,language_Italian,language_Japanese,language_Korean,language_Polish,language_Portuguese,language_Russian,language_Spanish,language_Swedish,language_Thai,language_Turkish
0,1,5,43,1,7.000000,1,-6.000000,0,0.142857,7,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,25,43,7,1.142857,49,5.857143,0,6.124946,2,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,37,43,4,2.500000,16,1.500000,0,1.599994,3,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,2,43,2,2.500000,4,-0.500000,0,0.799997,4,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,17,43,1,2.000000,1,-1.000000,0,0.499998,2,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194643,993,1,16,1,2.000000,1,-1.000000,0,0.499998,2,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
194644,993,4,16,1,3.000000,1,-2.000000,0,0.333332,3,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
194645,993,3,16,1,3.000000,1,-2.000000,0,0.333332,3,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
194646,993,10,16,3,2.666667,9,0.333333,0,1.124996,4,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## Models

### Unimodel Random Forest

In [5]:
from sklearn.ensemble import RandomForestClassifier
# import linear classifier
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

RandomForestClassifier()

In [6]:
prob_predictions = clf.predict_proba(X_test)
prob_predictions_df = pd.DataFrame(prob_predictions, columns=clf.classes_, index=X_test.index)
prob_predictions_df

,0,1
0,0.98,0.02
1,0.91,0.09
2,0.83,0.17
3,0.79,0.21
4,0.99,0.01
...,...,...
194643,1.00,0.00
194644,0.98,0.02
194645,0.98,0.02
194646,0.89,0.11


## Evaluating results

In [7]:
preds = pd.DataFrame({
    #language is the column that was one-hot encoded and has a 1 starting with language_
    'language': X_test.filter(like='language_').idxmax(axis=1).str.replace('language_', ''),
    'sentence_id': test['sentence_id'],
    'node': test['node'],
    'zero': prob_predictions[:, 0],
    'root_prob': prob_predictions[:, 1],
})
preds

,language,sentence_id,node,zero,root_prob
0,Japanese,1,5,0.98,0.02
1,Japanese,1,25,0.91,0.09
2,Japanese,1,37,0.83,0.17
3,Japanese,1,2,0.79,0.21
4,Japanese,1,17,0.99,0.01
...,...,...,...,...,...
194643,Russian,993,1,1.00,0.00
194644,Russian,993,4,0.98,0.02
194645,Russian,993,3,0.98,0.02
194646,Russian,993,10,0.89,0.11


In [19]:
preds_grouped = (
    preds.groupby(['language', 'sentence_id'], sort=False)
    .apply(lambda x: x.loc[[x['root_prob'].idxmax()]].assign(root=x.loc[x['root_prob'].idxmax(), 'node']))
    .reset_index(drop=True)
    .drop(columns=['root_prob','language','sentence_id','zero'])  # Optional: drop the old root_prob if no longer needed
)
preds_grouped
# Save the predictions to a CSV file with an id column from 1 to length of preds_grouped
preds_grouped['id'] = range(1, len(preds_grouped) + 1)
preds_grouped = preds_grouped[['id', 'root']]
preds_grouped
preds_grouped.to_csv('data/predictions_submission.csv', index=False)


/tmp/ipykernel_241403/98193650.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[[x['root_prob'].idxmax()]].assign(root=x.loc[x['root_prob'].idxmax(), 'node']))


In [20]:
current_predictions = pd.read_csv('data/predictions_submission.csv')
current_predictions.head()

,id,root
0,1,2
1,2,7
2,3,21
3,4,19
4,5,6


In [21]:
kaggle_perfect_predictions = pd.read_csv("data/kaggle_perfect_predictions.csv")
current_predictions = pd.read_csv('data/predictions_submission.csv')
kaggle_perfect_predictions = kaggle_perfect_predictions.drop(columns=['id'])
current_predictions = current_predictions.drop(columns=['id'])
current_predictions = current_predictions.drop(index=0)
kaggle_perfect_predictions = kaggle_perfect_predictions.drop(index=0)

print(classification_report(kaggle_perfect_predictions, current_predictions))

              precision    recall  f1-score   support

           1       0.29      0.32      0.30       690
           2       0.30      0.32      0.31       675
           3       0.33      0.34      0.33       687
           4       0.30      0.32      0.31       640
           5       0.33      0.32      0.33       693
           6       0.32      0.31      0.31       626
           7       0.29      0.26      0.27       653
           8       0.29      0.28      0.28       607
           9       0.27      0.26      0.27       547
          10       0.25      0.25      0.25       510
          11       0.28      0.29      0.28       491
          12       0.27      0.29      0.27       407
          13       0.27      0.24      0.25       390
          14       0.25      0.26      0.26       346
          15       0.27      0.25      0.26       336
          16       0.26      0.24      0.25       312
          17       0.28      0.27      0.27       248
          18       0.26    

/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: Undefine

In [22]:
def evaluate_model(y_true, y_pred):
        print("Print number of correct predictions:")
        correct_predictions = (y_true == y_pred).sum()
        final_score = correct_predictions / len(y_true)
        print(f"Evaluation accuracy: {final_score}")
evaluate_model(kaggle_perfect_predictions, current_predictions)

Print number of correct predictions:
Evaluation accuracy: root    0.281797
dtype: float64
